# Gaurav's Amazon Product Analytics
## IBM Data Analytics Project

**Project question:** What can product prices, discounts, ratings, review counts, and product categories tell us about the Amazon India product listings in this dataset?

**Tools:** Python, pandas, NumPy, Matplotlib, Seaborn  
**Dataset:** `data/amazon.csv` (Amazon product listing and review dataset)

### Project workflow
1. Load and inspect the data.
2. Clean price, discount, rating, and rating-count fields.
3. Check missing values and duplicates.
4. Explore category distribution, prices, discounts, ratings, and review counts.
5. Visualize patterns and write evidence-based recommendations.

> This is a descriptive analysis of the supplied dataset, not a causal study. Results describe the listings in this file and should not be treated as representative of all Amazon products or current prices.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)

# Works when the notebook is run from the project folder.
possible_paths = [
    Path("data/amazon.csv"),
    Path("amazon.csv"),
    Path("../data/amazon.csv"),
]
data_path = next((p for p in possible_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find amazon.csv. Keep the data folder beside this notebook, "
        "or update data_path to the location of your CSV."
    )

df = pd.read_csv(data_path)
print("Loaded:", data_path)
print("Shape:", df.shape)
df.head()

## 1. Initial inspection

In [ ]:
print("Rows, columns:", df.shape)
display(df.info())
display(df.describe(include="all").T.head(20))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate product IDs:", df["product_id"].duplicated().sum())

## 2. Data cleaning

Price and percentage fields contain symbols and separators, so convert them to numeric values. Preserve the original columns for traceability and create cleaned analysis columns.

In [ ]:
def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str).str.replace(r"[₹,%\s,]", "", regex=True),
        errors="coerce"
    )

data = df.copy()
for col in ["discounted_price", "actual_price", "discount_percentage", "rating", "rating_count"]:
    data[col + "_num"] = clean_numeric(data[col])

# Break the hierarchical category string into high-level and subcategory fields.
data["main_category"] = data["category"].astype(str).str.split("|").str[0].str.strip()
data["sub_category"] = data["category"].astype(str).str.split("|").str[1].str.strip()

data["savings_amount"] = data["actual_price_num"] - data["discounted_price_num"]
data["computed_discount_pct"] = np.where(
    data["actual_price_num"] > 0,
    data["savings_amount"] / data["actual_price_num"] * 100,
    np.nan
)

# Keep missing rating counts missing; use dropna in analyses that require them.
display(data[[
    "product_name", "discounted_price", "discounted_price_num",
    "actual_price", "actual_price_num", "discount_percentage_num",
    "rating_num", "rating_count_num", "main_category"
]].head())
print("Missing values after conversion:")
display(data[["discounted_price_num", "actual_price_num", "discount_percentage_num",
              "rating_num", "rating_count_num"]].isna().sum().to_frame("missing"))

## 3. Key performance indicators (KPIs)

In [ ]:
kpis = pd.Series({
    "Number of product listings": len(data),
    "Number of main categories": data["main_category"].nunique(),
    "Average product rating": data["rating_num"].mean(),
    "Median discounted price (₹)": data["discounted_price_num"].median(),
    "Average listed discount (%)": data["discount_percentage_num"].mean(),
    "Listings missing rating": data["rating_num"].isna().sum(),
    "Listings missing rating count": data["rating_count_num"].isna().sum(),
})
display(kpis.to_frame("value").round(2))

## 4. Category analysis

In [ ]:
category_summary = (
    data.groupby("main_category", dropna=False)
    .agg(
        product_count=("product_id", "count"),
        average_rating=("rating_num", "mean"),
        average_discount_pct=("discount_percentage_num", "mean"),
        median_discounted_price=("discounted_price_num", "median"),
        median_rating_count=("rating_count_num", "median"),
    )
    .sort_values("product_count", ascending=False)
)
display(category_summary.round(2))

top_categories = category_summary.head(10).sort_values("product_count")
plt.figure(figsize=(10, 6))
top_categories["product_count"].plot(kind="barh")
plt.title("Product listings by main category (top 10)")
plt.xlabel("Number of listings")
plt.ylabel("Main category")
plt.tight_layout()
plt.show()

## 5. Price and discount analysis

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=data, x="discounted_price_num", bins=40)
plt.xlim(0, data["discounted_price_num"].quantile(0.98))
plt.title("Distribution of discounted prices (up to 98th percentile)")
plt.xlabel("Discounted price (₹)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
sns.histplot(data=data, x="discount_percentage_num", bins=30)
plt.title("Distribution of listed discount percentages")
plt.xlabel("Listed discount (%)")
plt.tight_layout()
plt.show()

price_discount = data[["discounted_price_num", "actual_price_num",
                       "discount_percentage_num", "savings_amount"]].describe().T
display(price_discount.round(2))

## 6. Ratings and review engagement

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=data, x="rating_num", bins=20)
plt.title("Distribution of product ratings")
plt.xlabel("Rating")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
sns.scatterplot(data=data, x="rating_count_num", y="rating_num", alpha=0.55)
plt.xscale("log")
plt.title("Rating versus review count (log scale)")
plt.xlabel("Rating count (log scale)")
plt.ylabel("Rating")
plt.tight_layout()
plt.show()

rating_by_category = (
    data.groupby("main_category")
    .agg(product_count=("product_id", "count"), average_rating=("rating_num", "mean"))
    .query("product_count >= 5")
    .sort_values("average_rating", ascending=False)
)
display(rating_by_category.round(2))

## 7. Relationship between price, discount, ratings, and reviews

In [ ]:
numeric_cols = ["discounted_price_num", "actual_price_num",
                "discount_percentage_num", "rating_num", "rating_count_num"]
corr = data[numeric_cols].corr(numeric_only=True)
plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix for numeric fields")
plt.tight_layout()
plt.show()
display(corr.round(3))

## 8. Top listings for review

The tables below identify listings with the largest review counts and highest listed discounts. These are descriptive rankings, not endorsements. Always validate unusual prices or discount claims before making business decisions.

In [ ]:
cols = ["product_name", "main_category", "discounted_price_num",
        "actual_price_num", "discount_percentage_num", "rating_num", "rating_count_num"]

print("Top 10 by rating count")
display(data.sort_values("rating_count_num", ascending=False)[cols].head(10))

print("Top 10 by listed discount")
display(data.sort_values("discount_percentage_num", ascending=False)[cols].head(10))

## 9. Findings and recommendations

Use the tables and charts above to confirm the exact figures after running the notebook.

- **Assortment:** Compare listing counts across main categories before deciding where to focus product research.
- **Pricing:** Use median prices as well as averages because product prices are typically skewed by expensive outliers.
- **Discounts:** Compare listed discounts with the difference between actual and discounted price; unusually high discounts deserve validation.
- **Customer feedback:** Review count is a measure of review activity, not a direct measure of satisfaction. Read review content before interpreting a rating.
- **Comparisons:** Compare products within similar subcategories and price bands rather than treating all products as directly comparable.
- **Limitations:** This file is a snapshot, contains a small number of missing rating fields, and does not include sales volume, profit, inventory, or a collection date. Therefore it cannot establish causal effects or current market conditions.

## 10. Conclusion

This project demonstrates an end-to-end descriptive data analytics workflow: data inspection, cleaning, feature creation, exploratory analysis, visualization, and business recommendations using Python. The most useful next step would be to add time-series and actual sales/profit data so that changes and business impact can be measured.